[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part3_spatial_ai/ch17_open_world/01_foundation_models_for_slam.ipynb)

# 第17章: オープンワールド空間AI

本ノートブックでは、基盤モデル（Foundation Models）を活用した次世代のSLAMと3D理解について学びます。

## 学習内容
1. **オープンボキャブラリクエリ**: 事前定義されたカテゴリに限らず、自然言語で3Dマップを検索
2. **CLIPライクな特徴ベクトル**: テキストと画像の共有埋め込み空間のシミュレーション
3. **空間クエリデモ**: 「赤い椅子はどこ？」のような自然言語による空間検索

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
np.random.seed(42)

## 1. CLIP的な共有埋め込み空間の概念

CLIP (Contrastive Language-Image Pre-training) は、画像とテキストを同じベクトル空間に埋め込みます。これにより：
- 画像の特徴ベクトルとテキストの特徴ベクトルのコサイン類似度で関連性を測定
- 事前に定義されたカテゴリが不要（**オープンボキャブラリ**）
- 3Dマップの各点にCLIP特徴を付与すれば、自然言語で空間検索が可能

ここでは、この仕組みを簡略化してシミュレートします。

In [ ]:
# --- Simulated CLIP-like embedding space ---

class SimulatedCLIP:
    """
    Simulates a CLIP-like shared embedding space.
    In reality, CLIP uses large transformers. Here we use a simple
    concept-to-vector mapping to illustrate the principle.
    """
    def __init__(self, embed_dim=64):
        self.embed_dim = embed_dim
        # Define semantic concepts and their basis vectors
        self.concepts = {
            # Objects
            'chair': self._make_concept([1, 0, 0, 0, 0, 0, 0, 0]),
            'table': self._make_concept([0, 1, 0, 0, 0, 0, 0, 0]),
            'door': self._make_concept([0, 0, 1, 0, 0, 0, 0, 0]),
            'window': self._make_concept([0, 0, 0, 1, 0, 0, 0, 0]),
            'bookshelf': self._make_concept([0, 0, 0, 0, 1, 0, 0, 0]),
            'plant': self._make_concept([0, 0, 0, 0, 0, 1, 0, 0]),
            # Attributes
            'red': self._make_concept([0, 0, 0, 0, 0, 0, 1, 0]),
            'blue': self._make_concept([0, 0, 0, 0, 0, 0, 0, 1]),
            'wooden': self._make_concept([0, 1, 0, 0, 1, 0, 0, 0]),  # related to table+bookshelf
            'seating': self._make_concept([1, 0, 0, 0, 0, 0, 0, 0]),  # related to chair
            'furniture': self._make_concept([1, 1, 0, 0, 1, 0, 0, 0]),  # chair+table+bookshelf
        }
    
    def _make_concept(self, seed):
        """Create a full-dimensional embedding from a concept seed."""
        rng = np.random.RandomState(hash(tuple(seed)) % 2**31)
        base = np.array(seed + [0] * (self.embed_dim - len(seed)), dtype=float)
        # Add structured noise to simulate learned features
        base += rng.randn(self.embed_dim) * 0.1
        return base / np.linalg.norm(base)
    
    def encode_text(self, text):
        """Encode text query into embedding space (simplified)."""
        words = text.lower().replace(',', ' ').split()
        embedding = np.zeros(self.embed_dim)
        matched = 0
        for word in words:
            if word in self.concepts:
                embedding += self.concepts[word]
                matched += 1
        if matched == 0:
            return np.random.randn(self.embed_dim) * 0.1
        embedding /= np.linalg.norm(embedding)
        return embedding
    
    def encode_object(self, label, attributes=None):
        """Encode an object (with optional attributes) into embedding space."""
        embedding = np.zeros(self.embed_dim)
        if label in self.concepts:
            embedding += self.concepts[label]
        if attributes:
            for attr in attributes:
                if attr in self.concepts:
                    embedding += self.concepts[attr] * 0.5
        # Add observation noise
        embedding += np.random.randn(self.embed_dim) * 0.08
        embedding /= np.linalg.norm(embedding)
        return embedding

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Create CLIP model and encode objects
clip = SimulatedCLIP()

# 3D map objects with their embeddings
map_objects = [
    {'name': 'red chair', 'pos': [2, 3, 0], 'label': 'chair', 'attrs': ['red']},
    {'name': 'blue chair', 'pos': [4, 3, 0], 'label': 'chair', 'attrs': ['blue']},
    {'name': 'wooden table', 'pos': [3, 5, 0], 'label': 'table', 'attrs': ['wooden']},
    {'name': 'door', 'pos': [0, 4, 0], 'label': 'door', 'attrs': []},
    {'name': 'bookshelf', 'pos': [6, 7, 0], 'label': 'bookshelf', 'attrs': ['wooden']},
    {'name': 'window', 'pos': [3, 8, 0], 'label': 'window', 'attrs': ['blue']},
    {'name': 'plant', 'pos': [5, 2, 0], 'label': 'plant', 'attrs': []},
    {'name': 'small table', 'pos': [7, 4, 0], 'label': 'table', 'attrs': []},
]

# Encode all objects
for obj in map_objects:
    obj['embedding'] = clip.encode_object(obj['label'], obj['attrs'])

# Test queries
queries = [
    "red chair",
    "furniture",
    "wooden table",
    "blue",
    "door",
]

print("=== Open-Vocabulary Spatial Queries ===\n")
for query in queries:
    query_emb = clip.encode_text(query)
    scores = [(obj['name'], cosine_similarity(query_emb, obj['embedding'])) 
              for obj in map_objects]
    scores.sort(key=lambda x: -x[1])
    
    print(f"Query: \"{query}\"")
    for name, score in scores[:3]:
        print(f"  {name:15s}: similarity = {score:.3f}")
    print()

## 2. 空間クエリの可視化

テキストクエリに対する3Dマップ上の関連度をヒートマップとして可視化します。これにより、「赤い椅子はどこ？」のような自然言語での空間検索がどう機能するか直感的に理解できます。

In [ ]:
# --- Spatial query visualization ---

query_texts = ["red chair", "wooden furniture", "door", "blue"]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

obj_base_colors = {
    'chair': '#FF6B6B', 'table': '#8B4513', 'door': '#228B22',
    'window': '#4169E1', 'bookshelf': '#8B6914', 'plant': '#2E8B57'
}

for ax, query in zip(axes.flat, query_texts):
    query_emb = clip.encode_text(query)
    
    # Compute similarity scores
    scores = [cosine_similarity(query_emb, obj['embedding']) for obj in map_objects]
    scores = np.array(scores)
    # Normalize to [0, 1] for visualization
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    
    ax.set_xlim(-1, 9)
    ax.set_ylim(-1, 10)
    ax.set_aspect('equal')
    
    for i, obj in enumerate(map_objects):
        x, y = obj['pos'][0], obj['pos'][1]
        
        # Draw relevance halo
        circle = plt.Circle((x, y), 0.3 + scores_norm[i] * 0.8, 
                            color='yellow', alpha=scores_norm[i] * 0.6)
        ax.add_patch(circle)
        
        # Draw object
        color = obj_base_colors.get(obj['label'], 'gray')
        ax.scatter(x, y, c=color, s=120, zorder=5, edgecolors='black', linewidth=1.5)
        ax.annotate(f"{obj['name']}\n({scores[i]:.2f})", (x, y - 0.6),
                   ha='center', fontsize=8, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.8))
    
    # Highlight best match
    best_idx = np.argmax(scores)
    bx, by = map_objects[best_idx]['pos'][0], map_objects[best_idx]['pos'][1]
    ax.scatter(bx, by, s=300, facecolors='none', edgecolors='red', linewidth=3, zorder=6)
    
    ax.set_title(f'Query: "{query}"', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spatial_queries.png', dpi=100, bbox_inches='tight')
plt.show()
print("黄色のハロー = クエリとの関連度、赤丸 = 最もマッチする物体")

## 3. 埋め込み空間の構造

テキストとオブジェクトの埋め込みベクトルを2次元に射影し、共有空間の構造を可視化します。

In [ ]:
# --- Visualize embedding space with PCA-like projection ---

# Collect all embeddings
all_embeddings = []
all_labels = []
all_types = []  # 'object' or 'text'

for obj in map_objects:
    all_embeddings.append(obj['embedding'])
    all_labels.append(obj['name'])
    all_types.append('object')

text_queries = ["chair", "table", "furniture", "red", "blue", "wooden", "door"]
for q in text_queries:
    all_embeddings.append(clip.encode_text(q))
    all_labels.append(f'"{q}"')
    all_types.append('text')

all_embeddings = np.array(all_embeddings)

# Simple 2D projection using first 2 principal components
mean = np.mean(all_embeddings, axis=0)
centered = all_embeddings - mean
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
proj_2d = centered @ Vt[:2].T

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

for i, (pos, label, typ) in enumerate(zip(proj_2d, all_labels, all_types)):
    if typ == 'object':
        ax.scatter(pos[0], pos[1], s=100, c='steelblue', zorder=5, edgecolors='black')
        ax.annotate(label, (pos[0] + 0.02, pos[1] + 0.02), fontsize=9,
                   color='navy', fontweight='bold')
    else:
        ax.scatter(pos[0], pos[1], s=100, c='tomato', marker='D', zorder=5, edgecolors='black')
        ax.annotate(label, (pos[0] + 0.02, pos[1] - 0.04), fontsize=9,
                   color='darkred', fontstyle='italic')

# Draw similarity connections for one query
query_emb = clip.encode_text("furniture")
query_proj = (query_emb - mean) @ Vt[:2].T
for i, obj in enumerate(map_objects):
    sim = cosine_similarity(query_emb, obj['embedding'])
    if sim > 0.3:
        obj_proj = proj_2d[i]
        ax.plot([query_proj[0], obj_proj[0]], [query_proj[1], obj_proj[1]],
                'g--', alpha=sim, linewidth=sim * 3)

ax.set_title('Shared Embedding Space (PCA projection)\nBlue=Objects, Red=Text queries', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('embedding_space.png', dpi=100, bbox_inches='tight')
plt.show()
print("意味的に近い物体とテキストが埋め込み空間上で近くに配置されています")

## 演習問題

1. **新しいクエリ**: 「seating near window」のような複合クエリを実装してください。空間的な近接性も考慮するにはどうすればよいですか？
2. **ポイントレベル特徴**: 物体レベルではなく、3Dポイントクラウドの各点にCLIP特徴を付与する方式を設計してください。
3. **階層的クエリ**: 「キッチンにある赤いもの」のように、部屋レベル→物体レベルの階層的な検索を実装してください。

## まとめ

- **基盤モデル（CLIP等）**は、画像とテキストの共有埋め込み空間を学習し、オープンボキャブラリな認識を可能にする
- 3Dマップの各要素にCLIP特徴を付与することで、**自然言語による空間検索**が実現できる
- 実際のシステム（ConceptFusion, LERF, OpenScene等）では、NeRFや3DGSと統合して密な3D特徴場を構築する
- オープンワールド空間AIは、ロボットが未知の環境で柔軟にタスクを遂行するための基盤技術となる